# Basic eigenmode simulation of an X-Mon Qubit (Gmsh)

In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
os.environ["PMIX_MCA_gds"]="hash"

# Import useful packages
import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs
from qiskit_metal.toolbox_metal import math_and_overrides
from qiskit_metal.qlibrary.core import QComponent
from collections import OrderedDict

# To create plots after geting solution data.
import matplotlib.pyplot as plt
import numpy as np

# Packages for the simple design
from SQDMetal.Comps.Xmon import Xmon
from SQDMetal.Comps.Junctions import JunctionDolanPinStretch


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [6]:


# Set up chip design as planar, multiplanar also available
design = designs.DesignPlanar({}, overwrite_enabled=True)
gui = MetalGUI(design)

# Set up chip dimensions 
design.chips.main.size.size_x = '500um'
design.chips.main.size.size_y = '500um'
design.chips.main.size.size_z = '500um'
design.chips.main.size.center_x = '0mm'
design.chips.main.size.center_y = '0mm'



In [20]:
design.delete_all_components()
Xmon_left = Xmon(design, 'Xmon_left', options=Dict(pos_x=-0.2, pos_y=0,
                                    vBar_width='24um', hBar_width='24um', vBar_gap=f'{16}um', hBar_gap=f'{16}um',
                                    cross_width=f'{60*2+24}um', cross_height=f'{60*2+24}um',
                                    gap_up='24um', gap_left='24um', gap_right='24um', gap_down='24um'))


Xmon_right = Xmon(design, 'Xmon_right', options=Dict(pos_x=0.2, pos_y=0,
                                    vBar_width='24um', hBar_width='24um', vBar_gap=f'{16}um', hBar_gap=f'{16}um',
                                    cross_width=f'{60*4+24}um', cross_height=f'{60*4+24}um',
                                    gap_up='24um', gap_left='24um', gap_right='24um', gap_down='24um'))


junction_left = JunctionDolanPinStretch(design, 'junction_left', options=Dict(pin_inputs=Dict(start_pin=Dict(component=f'Xmon_left',pin='down')),
                                                         dist_extend='24um',
                                                         layer=2,
                                                         finger_width='0.4um', t_pad_size='0.385um',
                                                         squid_width='5.4um', prong_width='0.9um'));

junction_right = JunctionDolanPinStretch(design, 'junction_right', options=Dict(pin_inputs=Dict(start_pin=Dict(component=f'Xmon_right',pin='down')),
                                                         dist_extend='24um',
                                                         layer=2,
                                                         finger_width='0.4um', t_pad_size='0.385um',
                                                         squid_width='5.4um', prong_width='0.9um'));

# rebuild the GUI

gui.rebuild()
gui.autoscale()
# gui.screenshot()
# gui.main_window.close()
#

In [22]:
from SQDMetal.PALACE.Eigenmode_Simulation import PALACE_Eigenmode_Simulation
from SQDMetal.Utilities.Materials import MaterialInterface

design.rebuild()

#Eigenmode Simulation Options
user_defined_options = {
                "mesh_refinement":  0,                             #refines mesh in PALACE - essetially divides every mesh element in half
                "dielectric_material": "silicon",                  #choose dielectric material - 'silicon' or 'sapphire'
                "starting_freq": 5e9,                              #starting frequency in Hz 
                "number_of_freqs": 2,                              #number of eigenmodes to find
                "solns_to_save": 3,                                #number of electromagnetic field visualizations to save
                "solver_order": 2,                                 #increasing solver order increases accuracy of simulation, but significantly increases sim time
                "solver_tol": 1.0e-8,                              #error residual tolerance foriterative solver
                "solver_maxits": 100,                              #number of solver iterations
                "mesh_sampling": 130,                              #number of points to mesh along a geometry
                "fillet_resolution":12,                             #Number of vertices per quarter turn on a filleted path
                "palace_dir":"apptainer run ~/Documents/palace.sif",#"PATH/TO/PALACE/BINARY",
                "num_cpus": 16
                }

#Creat the Palace Eigenmode simulation
eigen_sim = PALACE_Eigenmode_Simulation(name ='TwoXmonTest',                              #name of simulation
                                        metal_design = design,                                      #feed in qiskit metal design
                                        sim_parent_directory = "",            #choose directory where mesh file, config file and HPC batch file will be saved
                                        mode = 'simPC',                                               #choose simulation mode 'HPC' or 'simPC'                                          
                                        meshing = 'GMSH',                                         #choose meshing 'GMSH' or 'COMSOL'
                                        user_options = user_defined_options,                        #provide options chosen above
                                        view_design_gmsh_gui = False,                               #view design in GMSH gui 
                                        create_files = True)                                        #create mesh, config and HPC batch files
eigen_sim.add_metallic(1)
eigen_sim.add_ground_plane()

#Add in the RF ports
eigen_sim.create_port_JosephsonJunction('junction_left', L_J=4.3e-9, C_J=10e-15)
eigen_sim.create_port_JosephsonJunction('junction_right', L_J=5.3e-9, C_J=10e-15)

# #Fine-mesh routed paths
eigen_sim.fine_mesh_around_comp_boundaries(['Xmon_left, Xmon_right'], min_size=14e-6, max_size=75e-6)

eigen_sim.setup_EPR_interfaces(metal_air=MaterialInterface('Aluminium-Vacuum'), substrate_air=MaterialInterface('Silicon-Vacuum'), substrate_metal=MaterialInterface('Silicon-Aluminium'))

eigen_sim.prepare_simulation()

eigen_sim.run()

>> /usr/lib64/mpich/bin/mpirun -n 16 /opt/palace/bin/palace-x86_64.bin TwoXmonTest.json

_____________     _______
_____   __   \____ __   /____ ____________
____   /_/  /  __ ` /  /  __ ` /  ___/  _ \
___   _____/  /_/  /  /  /_/  /  /__/  ___/
  /__/     \___,__/__/\___,__/\_____\_____/


--> Warning!
Output folder is not empty; program will overwrite content! (outputFiles)
Git changeset ID: v0.13.0-418-g10a1313f
Running with 16 MPI processes, 1 OpenMP thread
Device configuration: omp,cpu
Memory configuration: host-std
libCEED backend: /cpu/self/xsmm/blocked


--> Warning!
118 mesh faces with no associated boundary element exist on the domain boundary!

Added 123 elements in 2 iterations of local bisection for under-resolved interior boundaries
Added 409 duplicate vertices for interior boundaries in the mesh
Added 927 duplicate boundary elements for interior boundaries in the mesh
Added 118 boundary elements for exterior boundaries to the mesh
Added 751 boundary elements for material

ImportError: Can't determine version for zstandard

Use this to open the gmsh gui, then you can open your mesh file from the gui to view it

In [6]:
from SQDMetal.PALACE.Utilities.GMSH_Navigator import GMSH_Navigator

gmsh_nav = GMSH_Navigator(eigen_sim.path_mesh)
gmsh_nav.open_GUI()